# Build the recording metafile (`<task>.mat`)

Python version of `matlab/BuildRecordingMeta.m`. After **Kilosort** and **Phy curation**, this bundles one task block into
the lab's grab-and-go metafile (CC Lab Data Organization Standard, section 5):

| field | contents |
|---|---|
| `meta.info` | session id, subject, task, sampling rates, unit inclusion rule, trial counts, source files |
| `meta.spikes` | per-unit cluster id, depth, best channel, label, n_spikes, contamination, rate, spike times (s) |
| `meta.trials` | one row per **attempted** trial on the neural clock: success, target x/y, reward, target group |
| `meta.events` | trial-start / trial-attempt times and the full synced event-code stream |
| `meta.lfp` | placeholder (not yet added) |

## How to run
1. Fill in the `[meta]` section of the session config (`configs/<probe>/<session_id>.toml`).
2. Set `CONFIG` below (or the `CCLAB_CONFIG` environment variable) and run all cells.
3. Output: `<task folder>/<task>.mat`; a copy of the config + git commit goes to `<task folder>/processed/_runs/`.

The same thing from a terminal: `uv run cclab-preproc build-meta configs/<probe>/<session_id>.toml`


In [ ]:
# ============================ SETTINGS ============================
import os
CONFIG = os.environ.get("CCLAB_CONFIG", "configs/npx/venni_20250731_SEF.toml")
# ==================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cclab_preproc import load_config
from cclab_preproc import meta as M

cfg = load_config(CONFIG)
s = M.settings(cfg)
print(cfg.summary())

## 1. Check the inputs

In [ ]:
for key in ("ks_dir", "sync_events_file", "behavior_file", "out_file"):
    p = s[key]
    status = "-" if p is None else ("ok" if Path(p).exists() else "MISSING")
    print(f"{key:<17} {status:<8} {p}")

for f in ("spike_times.npy", "spike_clusters.npy", "cluster_info.tsv"):
    print(f"  ks_dir/{f:<20} {'ok' if (Path(s['ks_dir']) / f).exists() else 'MISSING'}")

## 2. Build and save

In [ ]:
meta = M.build(cfg)

## 3. Summary

In [ ]:
units = pd.DataFrame({
    "cluster_id": meta["spikes"]["cluster_ids"],
    "label": meta["spikes"]["ks_labels"],
    "depth": meta["spikes"]["depths"],
    "ch": meta["spikes"]["ch_best"],
    "n_spikes": meta["spikes"]["n_spikes"],
    "fr": meta["spikes"]["firing_rate"],
})
print(units["label"].value_counts().to_string())
units.head(10)

In [ ]:
trials = pd.DataFrame(meta["trials"])
print(f"{len(trials)} attempted trials, {int((trials.trial_success == 1).sum())} successful")
trials.groupby("target_group").trial_success.agg(["count", "mean"]).rename(columns={"mean": "success_rate"})

## 4. Sanity check: PSTHs around trial attempt

Successful trials only, split by target group, for the first few units. This is just a quick look that spikes and
trials are aligned — not an analysis.

In [ ]:
N_UNITS = 6            # how many units to plot
WINDOW = (-0.5, 1.5)   # seconds around the attempt event
BIN = 0.02             # seconds

ok = trials.trial_success == 1
align = trials.attempt_time[ok].to_numpy()
groups = trials.target_group[ok].to_numpy()

n = min(N_UNITS, len(units))
fig, axes = plt.subplots(1, n, figsize=(3 * n, 2.6), sharex=True, squeeze=False)
for ax, u in zip(axes[0], range(n)):
    rate, t = M.psth(np.asarray(meta["spikes"]["spike_times_cell"][u]), align, WINDOW, BIN)
    for g in np.unique(groups):
        ax.plot(t, rate[groups == g].mean(axis=0), lw=1, label=f"{int(g)}")
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"cluster {units.cluster_id[u]} ({units.label[u]})", fontsize=9)
    ax.set_xlabel("time from attempt (s)")
axes[0][0].set_ylabel("rate (Hz)")
axes[0][-1].legend(title="target", fontsize=7, title_fontsize=7, frameon=False)
plt.tight_layout()
plt.show()